In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [14]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

prompt = """
    You're a tool for generating product category names based on the tags for each category.
    
    CRITICAL CONSTRAINT (read first): across ALL generated category names, no word may
    appear more than once (case-insensitive; ignore connectors like "&", "and", "of", "for").
    This is the hardest requirement and takes priority over how natural or generic a name sounds.

    Input: a JSON object where each key is a category ID (string) and each value \
    is a list of tags describing that category.
    Example input:
    { "0": ["laptop", "monitor", "desk"], "1": ["puzzle", "drone", "lego"] }
    
    Procedure to follow:
    1. Go through the categories in order.
    2. Keep a running mental list of every word already used in a previous category name.
    3. Before finalizing each new name, check it against that list.
    4. If a natural word choice would collide, replace it with a more specific or \
    alternative term (synonym, subcategory term, etc.) instead of dropping it silently.
    5. Once all names are generated, double check the full set once more for \
    repeated words before returning the final JSON. If you find a repeat, rename \
    the offending categories.

    Example of collision handling:
    - Category 0 tags: ["desk", "office chair"] -> "Office Furniture"
    - Category 1 tags: ["stapler", "notebook"] -> here "Office" is already used, so \
    instead of "Office Supplies" use "Desk Supplies" or "Stationery"
    
    Rules:
    - Generate one short category name (2-4 words) per key, based on its tags.
    - Return ONLY a JSON object with the old key and the new category name as its value.
    - Keys in the output must match the input keys exactly (as strings).
    - Output language always English.
    Output example:
    { "0": "Electronics & furniture", "1": "Toys & gadgets", "2": 'Supplies' }
"""

system_prompt = { 'role': 'system', 'content': prompt }

def get_categories_cluster_labels(clusters, temperature=0):
    clusters_prompt = { 'role': 'user', 'content': clusters }
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[system_prompt, clusters_prompt],
        temperature=temperature,
    )

    return response.choices[0].message.content

In [4]:
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [5]:
from lib.data_load import load_data

df = await load_data()
print(f"\n{'CONCATENATED':<62} {df.shape[0]:>6,} rows x {df.shape[1]} cols")
df.sample(10)

1429_1.csv                                                     34,660 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv       5,000 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv 28,332 rows

CONCATENATED                                                   67,992 rows x 28 cols


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.userCity,reviews.userProvince,reviews.username,source_file,dateAdded,dateUpdated,primaryCategories,imageURLs,manufacturerNumber,sourceURLs
30469,AV1YE_muvKc47QAVgpwE,NaN,B00U3FPN4U,Amazon Fire Tv,"Back To College,College Electronics,College Tv...","848719057492,amazonfiretv/51454342,amazonfiret...",Amazon,2016-12-09T00:00:00.000Z,2017-09-20T05:35:56Z,"2017-08-25T22:22:06.968Z,2017-08-19T09:27:02.6...",...,NaN,NaN,jwcmac,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
21984,AVpfl8cLLJeJML43AE3S,"Echo (White),,,\r\nEcho (White),,,","B00L9EPT8O,B01E6AO69U",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...","echowhite/263039693056,echowhite/152558276095,...",Amazon,2017-05-17T00:00:00.000Z,NaN,"2017-09-28T00:00:00Z,2017-09-08T00:00:00Z,2017...",...,NaN,NaN,AshMoney,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
1049,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-02-17T00:00:00.000Z,2017-05-21T05:55:59Z,"2017-04-30T00:42:00.000Z,2017-06-07T09:03:00.000Z",...,NaN,NaN,Glorybound,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
21467,AVqVGWQDv8e3D1O-ldFr,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018SZT3BK,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",amazonfirehd88intablet16gbblackb018szt3bk6thge...,Amazon,2017-07-01T00:00:00.000Z,NaN,"2017-08-09T00:00:00Z,2017-08-06T00:00:00Z",...,NaN,NaN,tc2017,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
20511,AV1YnRtnglJLPUi8IJmV,"Kindle Voyage E-reader, 6 High-Resolution Disp...",B00OQVZDJM,Amazon,"Walmart for Business,Office Electronics,Tablet...","amazon/b00oqvzdjm,848719056099,amazonkindlepap...",Amazon,2017-01-27T00:00:00.000Z,2017-09-05T22:09:30Z,"2017-08-31T22:32:40.238Z,2017-08-02T19:54:43.4...",...,NaN,NaN,Madasheck,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
17171,AVqVGWLKnnc1JgDc3jF1,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",B018Y23MNM,Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T...",firekidseditiontablet7displaywifi16gbgreenkidp...,Amazon,2017-02-19T00:00:00.000Z,2017-06-21T07:38:36Z,"2017-06-04T02:18:01.873Z,2017-06-03T18:42:11.710Z",...,NaN,NaN,Xcountry,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
27681,AVpfl8cLLJeJML43AE3S,"Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,","B00L9EPT8O,B01E6AO69U",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...","echowhite/263039693056,echowhite/152558276095,...",Amazon,2016-06-22T00:00:00.000Z,NaN,"2017-09-28T00:00:00Z,2017-09-08T00:00:00Z,2017...",...,NaN,NaN,jhnj13,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
47097,AVpgNzjwLJeJML43Kpxn,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...","amazonbasics/hl002619,amazonbasicsaaaperforman...",AmazonBasics,2016-10-26T00:00:00.000Z,NaN,2017-08-28T00:00:00Z,...,NaN,NaN,ByJack,Datafiniti_Amazon_Consumer_Reviews_of_Amazon_P...,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,Health & Beauty,https://images-na.ssl-images-amazon.com/images...,HL-002619,"https://www.barcodable.com/upc/841710106442,ht..."
40103,AVpgNzjwLJeJML43Kpxn,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...","amazonbasics/hl002619,amazonbasicsaaaperforman...",AmazonBasics,2017-05-27T00:00:00.000Z,NaN,2017-08-28T00:00:00Z,...,NaN,NaN,ByMartin Cohen,Datafiniti_Amazon_Consumer_Reviews_of_Amazon_P...,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,Health & Beauty,https://images-na.ssl-images-amazon.com/images...,HL-002619,"https://www.barcodable.com/upc/841710106442,ht..."
50844,AVpe7xlELJeJML43ypLz,AmazonBasics AA Performance Alkaline Batteries...,"B00QWO9P0O,B01IB83NZG,B00MNV8E0C",Amazonbasics,"AA,AAA,Electronics Features,Health,Electronics...",amazonbasicsaaperformancealkalinebatteries48co...,AmazonBasics,2015-12-18T00:00:00.000Z,NaN,2017-06

In [6]:
from lib.data_products_clean import clean_data_frame

df = clean_data_frame(df)
print(df.isna().sum())
display(df.sample(10))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


name                 0
brand                0
asins                0
categories           0
manufacturer         0
reviews.date         0
reviews.username     0
primaryCategories    0
reviews.rating       0
reviews.text         0
reviews.title        0
all_categories       0
dtype: int64


,name,brand,asins,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title,all_categories
56398,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"B018SZT3BK,B01AHB9CN2",Fire Tablets Computers/Tablets Networking Elec...,Amazon,2017-02-20T00:00:00.000Z,PowerHungry,Electronics,4.0,tested item gave gift simple enough needs brow...,Makes great gift first time Tablet Users,Fire Tablets Computers/Tablets Networking Elec...
10471,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2016-01-08T00:00:00.000Z,JRog,,5.0,Great starter tablet media consumption games,Great little tablet,Fire Tablets Tablets Computers Tablets Tablets...
40111,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2016-07-30T00:00:00.000Z,ByPapaJ,Health Beauty,5.0,Fantastic bargain holding well months later go...,Bargain,AA AAA Health Electronics Health Household Cam...
59963,All-New Fire HD 8 Tablet Alexa 8 HD Display 16...,Amazon,B01J94YIT6,Fire Tablets Tablets Tablets Amazon Tablets Co...,Amazon,2018-01-17T00:00:00.000Z,Dallasgirl,Electronics,5.0,husband never smartphone touch screen laptop t...,Great tablet ages,Fire Tablets Tablets Tablets Amazon Tablets Co...
54300,Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB...,Amazon,B018Y226XO,Fire Tablets Learning Toys Toys Tablets Amazon...,Amazon,2016-12-19T00:00:00.000Z,yosi,Toys Games Electronics,5.0,great table kids parents control great gre,great,Fire Tablets Learning Toys Toys Tablets Amazon...
66483,Fire HD 8 Tablet Alexa 8 HD Display 16 GB Tang...,Amazon,B018T075DC,Fire Tablets Tablets Tablets Amazon Tablets Ge...,Amazon,2017-01-03T00:00:00.000Z,Rdog18,Electronics,4.0,Great product 10 year old son easy find wants,Great son,Fire Tablets Tablets Tablets Amazon Tablets Ge...
16536,Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB...,Amazon,B018Y23MNM,Tablets Fire Tablets Computers Tablets Tablets,Amazon,2016-12-10T00:00:00.000Z,maxie28,,5.0,Bought granddaughter loves,Great tablet kid,Tablets Fire Tablets Computers Tablets Tablets
19004,Amazon Kindle Paperwhite eBook reader 4 GB 6 m...,Amazon,B00OQVZDJM,Walmart Business Office Electronics Tablets Of...,Amazon,2016-09-12T00:00:00.000Z,enthusiast,,5.0,Love using easy download find books easy store...,Excellent Product,Walmart Business Office Electronics Tablets Of...
45649,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2016-10-30T00:00:00.000Z,ByJanna,Health Beauty,5.0,ordered twice Every gadget game son seems need...,recommend average user,AA AAA Health Electronics Health Household Cam...
16553,Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB...,Amazon,B018Y23MNM,Tablets Fire Tablets Computers Tablets Tablets,Amazon,2016-12-08T00:00:00.000Z,Lizs,,5.0,great kids 3yr old uses fallen cover really pr...,Great,Tablets Fire Tablets Computers Tablets Tablets


In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=42)

### Select best K cluster number for K_means training

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.sparse import hstack

vec_name = TfidfVectorizer(min_df=1)
vec_categories = TfidfVectorizer(min_df=2)

X_train_name = vec_name.fit_transform(train_df["name"])
X_train_categories  = vec_categories.fit_transform(train_df['all_categories'])

# Set more signaling on product name than his category
X = hstack([X_train_name * 2.0, X_train_categories * 1.0])

k_scores = {}
for k in range(2, min(8, len(train_df["name"]))):
    k_means = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = k_means.fit_predict(X)
    k_score = silhouette_score(X, labels)
    k_scores[k] = k_score
    print(f"k={k} -> inercia={k_means.inertia_:.2f}  silhouette={k_score:.3f}")

best_k = max(k_scores, key=k_scores.get)

k=2 -> inercia=121280.38  silhouette=0.326
k=3 -> inercia=98612.78  silhouette=0.386
k=4 -> inercia=85607.81  silhouette=0.399
k=5 -> inercia=74235.44  silhouette=0.446
k=6 -> inercia=62238.36  silhouette=0.505
k=7 -> inercia=54106.17  silhouette=0.564


In [9]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
train_df['cluster'] = kmeans.fit_predict(X)

In [10]:
tags = {}
for cluster in range(best_k):
    tags[cluster] = list(set(sorted(train_df[train_df['cluster'] == cluster]['all_categories'].sum().split())))

display(tags)

{0: ['Office',
  'eBook',
  'Accessories',
  'Learning',
  'Wi-Fi',
  'Networking',
  'E-readers',
  'Top',
  'Unlocked',
  'Generation',
  'Computers',
  '4GB',
  'ElectronicsComputers/Tablets',
  '3G',
  'Games',
  'E-Reader',
  'Touch',
  'Device',
  'Kindle',
  'ElectronicsFire',
  'Voyage',
  'Rated',
  'Amazon',
  'Movies',
  'Bags',
  'Cases',
  'ElectronicsAmazon',
  'ElectronicsTablets',
  'Covers',
  'ElectronicsWalmart',
  'Kids',
  'See',
  '4th',
  'Fire',
  'Music',
  'Tablets',
  '...',
  'iPad',
  'E-Readers',
  'ElectronicsComputers',
  'Electronics',
  'Toys',
  'Readers',
  'Business',
  'Computer',
  'Store',
  'Computers/Tablets',
  'Tech',
  'Features',
  'Walmart'],
 1: ['Chargers',
  'Robot',
  'Accessories',
  'Camera',
  'AA',
  'Supplies',
  'Camcorder',
  'BeautyAA',
  'Household',
  'Beauty',
  'Baby',
  'Health',
  'Electronics',
  'AAA',
  'Batteries',
  'Photo',
  'Check',
  'Personal',
  'Care'],
 2: ['Office',
  'eBook',
  'Accessories',
  'Wi-Fi',
  '

In [18]:
import json

openai_clusters_categories_response = get_categories_cluster_labels(json.dumps(tags))
clusters_categories = json.loads(openai_clusters_categories_response)
print(clusters_categories)

{'0': 'Digital Reading Devices', '1': 'Power & Personal Care', '2': 'Tablet Networking Gear', '3': 'Storage & Media Tablets', '4': 'Smart Home Hardware', '5': 'E-Book Media Products', '6': 'Battery & Health Essentials'}


In [21]:
train_df['category'] = train_df['cluster'].apply(lambda cluster: clusters_categories[str(cluster)])
display(train_df.sample(10))

,name,brand,asins,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title,all_categories,cluster,category
24523,Echo White Echo White,Amazon,"B00L9EPT8O,B01E6AO69U",Stereos Remote Controls Amazon Echo Audio Dock...,Amazon,2016-11-25T00:00:00.000Z,paulierizzo,,4.0,easy hook learning curve ask questions,fun use,Stereos Remote Controls Amazon Echo Audio Dock...,4,Smart Home Hardware
22123,Echo White Echo White,Amazon,"B00L9EPT8O,B01E6AO69U",Stereos Remote Controls Amazon Echo Audio Dock...,Amazon,2017-05-27T00:00:00.000Z,jmrc,,5.0,Amazon Echo great every day discover new things,Amazon Echo,Stereos Remote Controls Amazon Echo Audio Dock...,4,Smart Home Hardware
38206,All-New Fire HD 8 Tablet 8 '' HD Display Wi-Fi...,Amazon,B01AHB9CN2,Electronics iPad Tablets Tablets Fire Tablets ...,Amazon,2017-02-22T00:00:00.000Z,LadyG0726,Electronics,5.0,tablet Amazon offers tons free apps hundreds t...,Best Decision Ever Made,Electronics iPad Tablets Tablets Fire Tablets ...,3,Storage & Media Tablets
8464,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2017-03-15T00:00:00.000Z,Tony,,5.0,Awesome kids great price .... expensive one,Great tablet,Fire Tablets Tablets Computers Tablets Tablets...,2,Tablet Networking Gear
51668,AmazonBasics AA Performance Alkaline Batteries...,Amazon Basics,"B00QWO9P0O,B01IB83NZG,B00MNV8E0C",AA AAA Electronics Features Health Electronics...,AmazonBasics,2016-01-05T00:00:00.000Z,Bykrazedgurl17,Health Beauty,5.0,batteries ... work,Works,AA AAA Electronics Features Health Electronics...,6,Battery & Health Essentials
41861,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2016-05-31T00:00:00.000Z,ByAndris Vaskis,Health Beauty,5.0,Great batteries great price.minor side note- b...,Five Stars,AA AAA Health Electronics Health Household Cam...,1,Power & Personal Care
15259,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,B018Y225IA,Computers/Tablets Networking Tablets eBook Rea...,Amazon,2017-02-06T00:00:00.000Z,Dawn,,5.0,purchased mom computer person easy use,Easy use,Computers/Tablets Networking Tablets eBook Rea...,3,Storage & Media Tablets
62712,Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB...,Amazon,B018Y22C2Y,Computers Fire Tablets Electronics Features Co...,Amazon,2016-12-11T00:00:00.000Z,seniorcitizen,Electronics,4.0,used AMAZON Prime `` FreeTime '' app kids grea...,Good educational device parental controls,Computers Fire Tablets Electronics Features Co...,0,Digital Reading Devices
38693,All-New Fire HD 8 Tablet 8 '' HD Display Wi-Fi...,Amazon,B01AHB9CN2,Electronics iPad Tablets Tablets Fire Tablets ...,Amazon,2017-01-23T00:00:00.000Z,Augieh,Electronics,4.0,'s different things daughter plus read well,perfect 9 year old,Electronics iPad Tablets Tablets Fire Tablets ...,3,Storage & Media Tablets
10174,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2016-01-08T00:00:00.000Z,Trevo4,,5.0,Great kid beginning Cheap prices plays games,Kindle,Fire Tablets Tablets Computers Tablets Tablets...,2,Tablet Networking Gear
